# Hafta 8 — DFS / BFS Tekrarı + Açgözlü Algoritmalar
## (Week 8 — DFS / BFS Review + Greedy Algorithms)

**Ders:** Veri Yapıları ve Algoritmalar (Data Structures & Algorithms)
**Süre:** ~90 dakika
**Seviye:** 2.–3. sınıf

---

### 🎯 Hedefler (Learning Objectives)
Bu defterin sonunda öğrenci:

1. **BFS ve DFS**'yi hızlıca hatırlayıp ne zaman hangisinin kullanılacağını ayırt eder.
2. **Açgözlü (greedy)** yaklaşımın temel fikrini ("her adımda **o anki** en iyi seçimi yap") açıklayabilir.
3. **Greedy'nin işe yaradığı** klasik örnekleri (dizi maksimumu, sınıf çizelgeleme, kesirli sırt çantası) kodlayabilir.
4. **Greedy'nin başarısız olduğu** durumları (0/1 sırt çantası, TSP, küme kaplama) tanıyabilir — "yaklaşık (approximation) çözüm" kavramını anlar.
5. Bir problem için **greedy mı, BFS/DFS mı, başka bir teknik mi** karar verebilir.

### Ön koşullar (Prerequisites)
- Listeler, sözlükler (dict), kümeler (set)
- Yığın (stack) ve kuyruk (queue) operasyonları
- Big-O temelleri (`O(N)`, `O(N²)`, `O(N log N)`)
- Hafta 6: graf, BFS, DFS


## 1) Hızlı Özet (Quick Summary)

Bu hafta iki konuyu birleştiriyoruz:

**A. DFS / BFS tekrarı (review).** Hafta 6'da öğrendiğimiz graf gezintilerini (graph traversal) kısaca hatırlayacağız: kuyruk (queue) ile **enine** (breadth-first), yığın/özyineleme (stack/recursion) ile **derine** (depth-first).

**B. Açgözlü algoritmalar (greedy algorithms).** Her adımda, sadece **şu an** elindeki en iyi seçeneği seçen algoritmalar. Mantığı çok basit ama **bazen optimum'u verir, bazen sadece "yeterince iyi" yaklaşık çözüm** verir.

> 🍬 **Sezgi:** Greedy algoritma = şeker dükkânına giren çocuk. Önündeki en büyük şekeri kapar; daha büyük bir tane görünce eskisini bırakır, yenisini kapar. (Wengrow, 2020)

> ⚠️ **Önemli not:** "Açgözlü" kelimesi gerçek hayatta kötü çağrışım yapar; **algoritma bağlamında ise sıkça çok hızlı sonuç veren bir tekniktir.**


## 2) Terim Sözlüğü (Terminology Glossary)

| Türkçe | İngilizce | 1 satır anlam |
|---|---|---|
| Açgözlü algoritma | Greedy algorithm | Her adımda **o an** en iyi görünen seçimi yapar |
| Yerel optimum | Local optimum | Yakın çevrede en iyi olan çözüm |
| Küresel optimum | Global optimum | Tüm olası çözümler içinde en iyi olan |
| Yaklaşık algoritma | Approximation algorithm | Optimum'a yakın, "yeterince iyi" çözüm veren |
| Sezgisel | Heuristic | Garantisi olmayan, pratikte iyi çalışan kural |
| NP-Tam | NP-Complete | Bilinen hızlı (polinom zamanlı) çözümü olmayan zor problem sınıfı |
| Enine arama | Breadth-First Search (BFS) | Grafta seviye seviye gezme (kuyruk) |
| Derinine arama | Depth-First Search (DFS) | Grafta bir kolu sonuna kadar takip etme (yığın / özyineleme) |
| Sırt çantası problemi | Knapsack problem | Sınırlı kapasiteyle en yüksek değeri seçme |
| Küme kaplama | Set covering | En az alt kümeyle tüm öğeleri kapsama |
| Çizelgeleme | Scheduling | Sınırlı kaynak/zamana iş yerleştirme |


## 3) Hızlı Tekrar: BFS ve DFS (Review)

### Temel fark
- **BFS (Breadth-First Search / Enine arama):** Başlangıç düğümünden **dalga dalga** dışarı yayılır → **kuyruk (queue, FIFO)** kullanır. **En kısa yol (ağırlıksız grafta)** için idealdir.
- **DFS (Depth-First Search / Derinine arama):** Bir dalı **sonuna kadar** takip eder, çıkmaza girerse geri döner → **yığın (stack, LIFO)** veya **özyineleme (recursion)** kullanır. Yol var mı, döngü var mı, topolojik sıralama gibi sorularda iyidir.

### Görsel sezgi (mental model)
```
        Ayşe
       /    \
     Berk   Ceyda
     /  \      \
  Defne Elif  Furkan

BFS sırası : Ayşe → Berk, Ceyda → Defne, Elif, Furkan   (seviye seviye)
DFS sırası : Ayşe → Berk → Defne → (geri) → Elif → (geri) → Ceyda → Furkan
```


In [ ]:
# Hafta 6'dan tanıdık küçük sosyal ağ grafı
# (Turkish-localized social network graph from Week 6)

sosyal_ag = {
    "Ayşe":   ["Berk", "Ceyda"],
    "Berk":   ["Ayşe", "Defne", "Elif"],
    "Ceyda":  ["Ayşe", "Furkan"],
    "Defne":  ["Berk"],
    "Elif":   ["Berk"],
    "Furkan": ["Ceyda"],
}

from collections import deque

def bfs(graf, baslangic):
    '''Breadth-First Search — kuyruk (queue) ile enine gezinti.'''
    ziyaret = {baslangic}
    sira = []                       # gezilme sırası (visit order)
    kuyruk = deque([baslangic])
    while kuyruk:
        dugum = kuyruk.popleft()    # FIFO
        sira.append(dugum)
        for komsu in graf[dugum]:
            if komsu not in ziyaret:
                ziyaret.add(komsu)
                kuyruk.append(komsu)
    return sira

def dfs(graf, baslangic, ziyaret=None, sira=None):
    '''Depth-First Search — özyineleme (recursion) ile derinine gezinti.'''
    if ziyaret is None: ziyaret, sira = set(), []
    ziyaret.add(baslangic)
    sira.append(baslangic)
    for komsu in graf[baslangic]:
        if komsu not in ziyaret:
            dfs(graf, komsu, ziyaret, sira)
    return sira

print("BFS:", bfs(sosyal_ag, "Ayşe"))
print("DFS:", dfs(sosyal_ag, "Ayşe"))


### 📌 Hatırlanacaklar (What to remember — DFS/BFS)
- **Veri yapısı = davranış:** Kuyruk → BFS, Yığın → DFS.
- **Ağırlıksız grafta en kısa yol** istiyorsan **BFS**.
- **Bağlantı var mı? / Tüm yolları gez** istiyorsan **DFS** yeterli.
- Her ikisi de **O(V + E)** karmaşıklığa sahiptir (V: düğüm sayısı, E: kenar sayısı).


## 4) Açgözlü Algoritmalar: Temel Fikir (Greedy — Basic Idea)

### Tanım
**Açgözlü algoritma**, **her adımda o anda en iyi görünen seçeneği** seçen ve geri dönüp yeniden düşünmeyen algoritmadır.

> "Each step chooses what appears to be the best option at that moment in time." — Wengrow, *A Common-Sense Guide to Data Structures and Algorithms* (2020)

### Artıları / Eksileri (Pros / Cons)
| ✅ Artı | ❌ Eksi |
|---|---|
| Yazımı **kolay** | Her zaman **optimum** vermez |
| Genelde **çok hızlı** (sıkça O(N) veya O(N log N)) | Bazı problemlerde **yanlış cevap** verebilir |
| **Az bellek** kullanır (genelde O(1) ek alan) | Doğru çalıştığını **kanıtlamak** gerekir |

### 🚫 Yaygın yanılgılar (Common pitfalls)
1. **"Greedy hep en hızlısı = hep en iyisi"** ❌ — Sadece bazı problemlerde optimumdur (örn. dizi maksimumu, sınıf çizelgeleme).
2. **"Çalışıyor gibi görünüyor"** ❌ — Birkaç örnekte doğru çıkması yetmez; **karşı örnek (counter-example)** aramak gerekir.
3. **"Greedy = sezgisel (heuristic)"** ❌ — Sezgiseller de greedy olabilir ama her greedy yaklaşım sezgisel değildir. Bazı greedy algoritmalar kanıtlı optimumdur (Dijkstra, Huffman).


## 5) İlk Örnek: Dizi Maksimumu (Array Max)

> **Problem:** Bir tam sayı dizisinde **en büyük sayıyı** bul.

**Naif yöntem:** İç içe iki döngü, her sayıyı diğer her sayıyla kıyasla → **O(N²)**.
**Sıralama ile:** Diziyi sırala, sonuncuyu döndür → **O(N log N)**.
**Greedy yaklaşımı:** İlk sayıyı "şimdilik en büyük" varsay; daha büyüğünü görünce güncelle → **O(N)** ✨


In [ ]:
def maksimum_acgozlu(dizi):
    '''Dizideki en büyük sayıyı greedy yaklaşımla bulur. O(N) zaman, O(1) bellek.'''
    en_buyuk = dizi[0]              # şimdilik en iyi tahmin (greedy assumption)
    for sayi in dizi[1:]:
        if sayi > en_buyuk:         # daha iyisini gördük → güncelle
            en_buyuk = sayi
    return en_buyuk

# Hızlı test (asserts)
assert maksimum_acgozlu([3, 1, 7, 2, 5]) == 7
assert maksimum_acgozlu([-4, -1, -9])    == -1
assert maksimum_acgozlu([42])            == 42
print("✅ Dizi maksimum testleri geçti.")


**Neden çalışıyor?** Dizideki en büyük sayı kesin olarak dizidedir; tek bir geçişte onunla karşılaşacağız ve o anda `en_buyuk`'ten daha büyük olacağı için güncelleyeceğiz. Yani **greedy adım yerel olarak optimal**, ama burada **küresel olarak da** doğru sonucu veriyor.


## 6) Klasik Örnek 1 — Sınıf Çizelgeleme (Class Scheduling)

> **Problem:** Tek bir derslikte, başlangıç–bitiş saatleri verilen derslerden **en çoğunu** çakışmadan yerleştir. (Bharya & Sanders, *Grokking Algorithms*, 8. Bölüm)

**Greedy kuralı:** "**En erken biten** dersi seç, çakışanları at, kalanlar arasından yine en erken biteni seç…"

| Ders | Başlangıç | Bitiş |
|---|---|---|
| Matematik | 09:00 | 10:00 |
| İngilizce | 09:30 | 10:30 |
| Sanat     | 10:00 | 11:00 |
| Tarih     | 10:30 | 11:30 |
| Müzik     | 11:00 | 12:00 |


In [ ]:
def cizelgele(dersler):
    '''En erken biten dersi seç-stratejisi (greedy by earliest end-time).
    dersler: [(ad, baslangic, bitis), ...]    saatler 24 saat formatında ondalık (örn 9.5)
    '''
    # 1) Bitiş saatine göre sırala  → O(N log N)
    sirali = sorted(dersler, key=lambda d: d[2])

    secilen = []
    son_bitis = 0
    for ad, bas, bit in sirali:
        if bas >= son_bitis:        # şu anki son dersle çakışmıyor mu?
            secilen.append(ad)
            son_bitis = bit
    return secilen

program = [
    ("Matematik", 9.0,  10.0),
    ("İngilizce", 9.5,  10.5),
    ("Sanat",     10.0, 11.0),
    ("Tarih",     10.5, 11.5),
    ("Müzik",     11.0, 12.0),
]
print("Seçilen dersler:", cizelgele(program))
# Beklenen: ['Matematik', 'Sanat', 'Müzik']  → 3 ders


### Adım adım çalışma (Step-by-step)

```
Sıralı (bitişe göre):  Matematik(09-10), Sanat(10-11), İngilizce(09:30-10:30), Müzik(11-12), Tarih(10:30-11:30)

Adım 1: Matematik seç     → son_bitis = 10:00     ✓
Adım 2: Sanat   (10≥10)  → son_bitis = 11:00     ✓
Adım 3: İngilizce(09:30<11) → çakışır, atla   ✗
Adım 4: Müzik   (11≥11)  → son_bitis = 12:00     ✓
Adım 5: Tarih   (10:30<12) → çakışır, atla   ✗

Sonuç: 3 ders (Matematik, Sanat, Müzik) — kanıtlanabilir biçimde optimum!
```

**Bu greedy neden optimum?** En erken biten dersi seçmek **kalan zamanı en fazla bırakır**. Kanıt için "exchange argument" tekniği kullanılır (algoritma derslerinde sıkça).


## 7) Klasik Örnek 2 — Kesirli Sırt Çantası (Fractional Knapsack)

> **Problem:** Kapasitesi `W` kg olan sırt çantası. Elimizde `(ağırlık, değer)` çiftleri olan eşyalar var. Her eşyadan **parça** alabiliriz (yarısını da alabiliriz — örn. altın tozu, tahıl). Toplam değeri **maksimize** et.

**Greedy kuralı:** En yüksek **birim ağırlık başına değer (₺/kg)** olan eşyadan başla; çanta dolana kadar al.

> ⚠️ **Önemli:** Bu greedy **kesirli** sırt çantasında **optimum**'dur. **0/1 sırt çantası**nda (parça alamadığımız versiyon) ise **işe yaramaz** — onu birazdan göreceğiz.


In [ ]:
def kesirli_sirt_cantasi(esyalar, kapasite):
    '''Greedy: en yüksek değer/ağırlık oranlı eşyadan başla.
    esyalar: [(ad, agirlik_kg, deger_TL), ...]
    Geri döndürür: (toplam_deger, alinan_listesi)
    '''
    # Oran hesapla ve büyükten küçüğe sırala
    sirali = sorted(esyalar, key=lambda e: e[2]/e[1], reverse=True)

    toplam_deger = 0.0
    alinan = []
    kalan = kapasite
    for ad, agirlik, deger in sirali:
        if kalan == 0:
            break
        alinacak = min(agirlik, kalan)               # tamamı sığar mı, sığmazsa kalan kadar
        oran     = alinacak / agirlik
        toplam_deger += deger * oran
        alinan.append((ad, round(alinacak, 2), round(deger*oran, 2)))
        kalan -= alinacak
    return toplam_deger, alinan

esyalar = [
    ("Altın tozu",  10, 600),   # 60 ₺/kg
    ("Gümüş",       20, 500),   # 25 ₺/kg
    ("Bakır",       30, 240),   # 8  ₺/kg
]
toplam, alindi = kesirli_sirt_cantasi(esyalar, kapasite=25)
print(f"Toplam değer: {toplam:.2f} ₺")
print("Alınanlar:", alindi)


**Çıktıyı yorumla:** 10 kg altın tozu (=600 ₺) + 15 kg gümüş (=375 ₺) = **975 ₺**. Çanta dolu (25 kg). Bakıra hiç sıra gelmedi çünkü en düşük oranlıydı. Bu, greedy'nin işe yaradığı temiz bir örnek.


## 8) Greedy Ne Zaman **Yetersiz** Kalır? — 0/1 Sırt Çantası ve TSP

### 0/1 sırt çantası
Aynı problem, ama eşyaların **parçası alınamaz** (ya hep ya hiç). Bu artık **NP-tam (NP-complete)** kabul edilen bir problemdir.


In [ ]:
# Karşı örnek (counter-example) — greedy burada yanlış cevap verir
esyalar_01 = [
    ("A", 10, 60),   # 6.0 ₺/kg  ← greedy bunu seçer
    ("B", 20, 100),  # 5.0 ₺/kg
    ("C", 30, 120),  # 4.0 ₺/kg
]
kapasite = 50

# Greedy (oran sırasıyla):
secim = []
kalan = kapasite
toplam = 0
for ad, w, v in sorted(esyalar_01, key=lambda e: e[2]/e[1], reverse=True):
    if w <= kalan:
        secim.append(ad); toplam += v; kalan -= w

print(f"Greedy seçim : {secim}, değer = {toplam}")    # A + B = 160
print(f"Optimum (B+C): değer = {100 + 120}")          # 220 — daha iyi!


**Sonuç:** Greedy 160 ₺ buldu, optimum 220 ₺. **0/1 sırt çantası için greedy yanlıştır.** Doğru çözüm: **dinamik programlama (dynamic programming)** veya geri izleme (backtracking).

### Gezgin Satıcı Problemi (Traveling Salesperson Problem, TSP)
> N şehirden geçen **en kısa** turu bul. Tam çözüm O(N!) zaman alır — N=20'de bile pratik değil.

**Greedy (sezgisel) yaklaşım:** "Her zaman **en yakın** ziyaret edilmemiş şehre git." Bu **yaklaşık** bir çözümdür — optimum'dan %25'e kadar uzaklaşabilir, ama N=1000 için **saniyeler** içinde bir tur verir.

> 💡 **Anahtar fikir:** NP-tam problemlerde greedy çoğu zaman **tek pratik yol**dur. "Mükemmel" yerine "**yeterince iyi**" demek zorundayız.


## 9) Klasik Örnek 3 — Küme Kaplama / Radyo İstasyonları (Set Covering)

> **Problem:** (Grokking 8. Bölüm) Türkiye'nin tüm illerine ulaşmak istiyoruz. Elimizde radyo istasyonları var; her istasyon **bazı illere** yayın yapıyor. **En az** kaç istasyon seçersek tüm iller kapsanır?

Bu problem **NP-tam**'dır. Greedy bize **yaklaşık** ama hızlı bir çözüm verir.

**Greedy kuralı:** "Her adımda, **henüz kapsanmamış illerden en çoğunu** kapsayan istasyonu seç."


In [ ]:
def set_kapla(hedef_iller, istasyonlar):
    '''Set covering — greedy approximation.
    hedef_iller : set, kapsanması gereken iller
    istasyonlar : dict {istasyon_adi: set_of_iller}
    '''
    kalan = set(hedef_iller)
    secilen = []
    while kalan:
        # Her adımda kalan illerin EN ÇOĞUNU kapsayan istasyonu bul
        en_iyi, en_iyi_kapsama = None, set()
        for ad, kapsama in istasyonlar.items():
            ortak = kapsama & kalan
            if len(ortak) > len(en_iyi_kapsama):
                en_iyi, en_iyi_kapsama = ad, ortak
        if en_iyi is None:
            break                  # hiçbir istasyon yeni il kapsamıyor
        secilen.append(en_iyi)
        kalan -= en_iyi_kapsama
    return secilen, kalan          # ikinci dönüş: kapsanamayan iller (varsa)

# Mini örnek: 5 il, 5 istasyon
hedef = {"İstanbul", "Ankara", "İzmir", "Bursa", "Antalya"}
istasyonlar = {
    "Marmara FM": {"İstanbul", "Bursa"},
    "Başkent FM": {"Ankara", "Bursa"},
    "Ege FM":     {"İzmir", "Antalya"},
    "Akdeniz FM": {"Antalya"},
    "Türkiye FM": {"İstanbul", "Ankara", "İzmir"},   # 3 il birden!
}
secim, eksik = set_kapla(hedef, istasyonlar)
print("Seçilen istasyonlar:", secim)
print("Kapsanamayan iller:", eksik)


**Adım adım:** İlk turda **Türkiye FM** 3 il kapsar (en çok) → seçilir. Kalan: {Bursa, Antalya}. İkinci turda Marmara FM (Bursa) veya Ege FM (Antalya+İzmir ama İzmir zaten kapsandı) seçilir. Bu **yaklaşık optimum**'dur; gerçek optimum bazen aynı, bazen 1 istasyon daha az olabilir.

### Karmaşıklık (Set covering greedy)
- **N** istasyon, **M** öğeli evren için: her döngüde tüm istasyonları kıyaslıyoruz → **O(N² · M)**.
- En kötü senaryoda greedy çözüm **ln(M)** çarpı optimum kadar büyüktür (kanıtlı yaklaşım garantisi).


## 10) Görsel — Greedy vs Optimum (Local vs Global Optimum)

Greedy'nin neden bazen **küresel optimum'u kaçırdığını** anlamak için tek değişkenli bir fonksiyon düşünelim. Greedy "yokuş yukarı tırman" yaparsa **ilk tepeyi** bulur, en yüksek tepeyi değil.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 10, 500)
# İki tepesi olan bir fonksiyon — soldaki yerel, sağdaki küresel optimum
y = np.sin(x) + 0.7*np.sin(2.3*x) + 0.3*x

plt.figure(figsize=(8, 4))
plt.plot(x, y, label="Hedef fonksiyon (objective)")

# Greedy "hill climbing" 1.5'ten başlasa nereye varır?
i = np.argmin(np.abs(x - 1.5))
while i+1 < len(x) and y[i+1] > y[i]:
    i += 1
plt.scatter(x[i], y[i], color="red",   zorder=5, s=80, label=f"Yerel optimum (greedy buldu) ≈ {y[i]:.2f}")

# Gerçek küresel optimum
j = np.argmax(y)
plt.scatter(x[j], y[j], color="green", zorder=5, s=80, label=f"Küresel optimum ≈ {y[j]:.2f}")

plt.title("Greedy yerel tepede sıkışabilir (Greedy can get stuck at a local optimum)")
plt.xlabel("x"); plt.ylabel("değer")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()


## 11) Karmaşıklık Analizi (Complexity Summary)

| Algoritma | Zaman | Bellek | Optimum mu? |
|---|---|---|---|
| Dizi maksimumu (greedy) | O(N) | O(1) | ✅ Evet |
| Sınıf çizelgeleme (greedy) | O(N log N) | O(1) | ✅ Evet (kanıtlı) |
| Kesirli sırt çantası | O(N log N) | O(1) | ✅ Evet |
| 0/1 sırt çantası **greedy** | O(N log N) | O(1) | ❌ Hayır (yaklaşık) |
| 0/1 sırt çantası **DP** | O(N · W) | O(N · W) | ✅ Evet |
| Set covering greedy | O(N² · M) | O(M) | ❌ Yaklaşık (ln M çarpan) |
| TSP greedy (en yakın komşu) | O(N²) | O(N) | ❌ Yaklaşık |
| Dijkstra (greedy + priority queue) | O((V+E) log V) | O(V) | ✅ Evet (ağırlıklar ≥ 0 ise) |
| BFS / DFS | O(V + E) | O(V) | — (gezinti) |

**Genel kural:** Greedy hızlıdır; ama **doğruluğunu kanıtlamadan** kullanma. Karşı örnek bulamıyorsan ve birkaç farklı girdiyi geçiyorsa muhtemelen iyidir — yine de matematiksel argüman istenebilir.


## 12) Gerçek Dünya Kullanımları (Real-world)

1. **Ağ yönlendirme (network routing) ve haritalar (maps).** Dijkstra algoritması greedy mantığıyla çalışır: her adımda "en kısa mesafedeki ziyaret edilmemiş düğümü" seç. Google Maps / Yandex Navigation / OpenStreetMap'in altında bu fikrin türevleri vardır.
2. **Sıkıştırma (compression) — Huffman kodlama.** ZIP, JPEG, MP3 gibi formatlar Huffman ağacını **en küçük iki frekansı birleştir** greedy kuralıyla kurar. Optimumdur ve **kanıtlıdır**.
3. **CPU çizelgeleme (OS scheduling) ve veri merkezleri.** "En kısa iş önce" (Shortest Job First), "en erken bitiş zamanı" gibi greedy kurallar işletim sistemlerinde ve bulut iş yükü yöneticilerinde kullanılır.


## 13) Mini Alıştırmalar (Exercises)

> Çözümler dosyanın sonundaki **"Çözümler"** bölümündedir. Önce kendin dene!

**Alıştırma 1 — Para üstü (Coin change, Türk parası).**
Türk parası: 200, 100, 50, 20, 10, 5, 1 kuruş. Verilen bir tutarı (örn. 287 kuruş) **en az sayıda madeni para ile** ver. Greedy bir fonksiyon yaz: her seferinde **kalan tutardan küçük veya eşit en büyük** kupürü seç.

> 💡 *İpucu:* Türk lirası kupürleri "canonical" yani greedy burada **optimum**'dur. Ama farklı kupür sistemlerinde (örn. {1, 3, 4} ile 6 yap) greedy yanlış cevap verebilir.

**Alıştırma 2 — En çok aktivite (Activity selection).**
Sana etkinlik listesi `[(ad, baslangic, bitis), ...]` veriliyor. Sınıf çizelgeleme fonksiyonunu kullanıp **kaç etkinliğin sığdığını** ve **hangilerinin** seçildiğini döndüren bir fonksiyon yaz. Aşağıdaki test girdisiyle çalıştır:
```python
[("Yoga", 6, 7), ("Toplantı", 6.5, 8), ("Ders", 7, 9),
 ("Sunum", 8.5, 9.5), ("Spor", 9, 10)]
```

**Alıştırma 3 — Karşı örnek bul (Find a counter-example).**
Aşağıdaki "greedy" algoritma her zaman doğru mu çalışır? **3 sayı seçimi:** Bir tamsayı dizisinden **çarpımları maksimum** olan **3 sayıyı** bulmak için "en büyük 3 sayıyı seç" greedy stratejisi yeter mi? **Karşı örnek** ver veya doğruluğunu argümante et.

> 💡 *İpucu:* Negatif sayıları düşün.


## 14) Kısa Quiz (5 soru)

**Q1 (çoktan seçmeli).** Aşağıdakilerden hangisi **kesinlikle** açgözlü bir algoritmadır?
- A) Quicksort
- B) Bubble Sort
- C) Dijkstra'nın en kısa yol algoritması
- D) Binary Search

**Q2 (doğru/yanlış).** Greedy algoritma her zaman optimum sonucu verir. (D/Y)

**Q3 (kısa cevap).** Sınıf çizelgeleme probleminde greedy stratejisi nedir? (Bir cümle)

**Q4 (çoktan seçmeli).** **0/1 sırt çantası** problemi için greedy çözüm:
- A) Optimumdur ve O(N) zaman alır
- B) Optimum **değildir**, dinamik programlama gerekir
- C) Sadece eşyalar tamsayı olduğunda çalışır
- D) Hiç çalışmaz, hata verir

**Q5 (doğru/yanlış).** BFS, ağırlıksız (unweighted) bir grafta iki düğüm arasında **en kısa yolu** garanti eder. (D/Y)


## 15) Tarihçe — Greedy Algoritmaların Kaynağı (History)

- **"Greedy algorithm" terimi** algoritma literatüründe **1970'ler**de yaygınlaştı; ancak fikir çok daha eskidir. **Edsger W. Dijkstra (1959)** en kısa yol algoritmasını yayımladığında, modern anlamda en etkili greedy algoritmalardan birini sunmuş oldu.
- **Huffman kodlama** — **David A. Huffman**, 1952'de MIT'de yüksek lisans öğrencisiyken hocası Robert Fano'nun sınıf projesi olarak verdiği problemi çözerken bu greedy algoritmayı buldu. Sınıf ödevini "ya bitir ya finale gir" diye yapan bir öğrencinin keşfi günümüzdeki çoğu dosya sıkıştırmanın temelinde duruyor.
- **Matroid teorisi** — **Jack Edmonds**'un 1971'deki çalışması, "**hangi problemlerde greedy her zaman optimumdur?**" sorusuna matematiksel cevap verdi (matroid yapılarına sahip problemlerde).

> Bazı ek detaylar için **kesin kaynak bulamadığım** noktalar var; konuyu derinleştirmek isteyen öğrenciler aşağıdaki kaynaklara başvurabilir.

**Linkler:**
- https://en.wikipedia.org/wiki/Greedy_algorithm
- https://en.wikipedia.org/wiki/Huffman_coding
- https://en.wikipedia.org/wiki/Edsger_W._Dijkstra


## 16) Kaynakça (References)

1. **Bhargava, A.** (2016). *Grokking Algorithms*. Manning. (Bölüm 8 — Greedy Algorithms) — https://www.manning.com/books/grokking-algorithms
2. **Wengrow, J.** (2020). *A Common-Sense Guide to Data Structures and Algorithms (2nd ed.)*. Pragmatic Bookshelf. (Bölüm 20 — Techniques for Code Optimization) — https://pragprog.com/titles/jwdsal2/a-common-sense-guide-to-data-structures-and-algorithms-second-edition/
3. **Cormen, T.H., Leiserson, C.E., Rivest, R.L., Stein, C.** (2022). *Introduction to Algorithms (4th ed.)*. MIT Press. (Bölüm 15 — Greedy Algorithms) — https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/
4. **MIT OpenCourseWare 6.006** — Introduction to Algorithms (free lectures) — https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-spring-2020/
5. **Princeton — R. Sedgewick & K. Wayne** *Algorithms* (free site/book) — https://algs4.cs.princeton.edu/


## 17) Çözümler (Solutions)

### Alıştırma 1 — Para üstü (Coin change)


In [ ]:
def para_ustu(tutar_kurus, kupurler=(200, 100, 50, 20, 10, 5, 1)):
    '''Türk kuruş kupürleriyle en az sayıda madeni para. Bu kupür kümesi 'canonical',
    yani greedy burada optimumdur.'''
    secim = []
    for k in kupurler:           # büyükten küçüğe
        while tutar_kurus >= k:
            secim.append(k)
            tutar_kurus -= k
    return secim

cozum = para_ustu(287)
print("287 kuruş için:", cozum)
print("Toplam parça :", len(cozum))
# Beklenen: [200, 50, 20, 10, 5, 1, 1] → 7 parça

# Karşı örnek: {1, 3, 4} ile 6 yapmak — greedy 4+1+1=3 parça verir, optimum 3+3=2 parça!
print("Karşı örnek :", para_ustu(6, kupurler=(4, 3, 1)))   # [4, 1, 1] — yanlış!


### Alıştırma 2 — En çok aktivite

In [ ]:
def aktivite_sec(etkinlikler):
    sirali = sorted(etkinlikler, key=lambda e: e[2])    # bitişe göre sırala
    secilen, son_bitis = [], 0
    for ad, bas, bit in sirali:
        if bas >= son_bitis:
            secilen.append(ad)
            son_bitis = bit
    return len(secilen), secilen

veri = [("Yoga", 6, 7), ("Toplantı", 6.5, 8), ("Ders", 7, 9),
        ("Sunum", 8.5, 9.5), ("Spor", 9, 10)]
sayi, hangi = aktivite_sec(veri)
print(f"Sığan etkinlik sayısı : {sayi}")
print(f"Seçilenler            : {hangi}")
# Beklenen: 3 — ['Yoga', 'Sunum', 'Spor']


### Alıştırma 3 — Çarpımı maksimum 3 sayı (counter-example)

**"En büyük 3 sayıyı seç" greedy YANLIŞTIR.** Karşı örnek:

`[-10, -9, 1, 2, 3]`
- Greedy → `1 × 2 × 3 = 6`
- **Optimum** → `(-10) × (-9) × 3 = 270`  ← iki negatifin çarpımı pozitif, en büyük pozitifle çarpılır.

**Doğru strateji:** "**En büyük 3 sayı**" vs **"En küçük 2 sayı × en büyük 1 sayı"** — hangisi büyükse onu döndür.


In [ ]:
def en_buyuk_carpim_3(dizi):
    s = sorted(dizi)
    aday1 = s[-1] * s[-2] * s[-3]               # en büyük 3
    aday2 = s[0]  * s[1]  * s[-1]               # en küçük 2 + en büyük 1
    return max(aday1, aday2)

assert en_buyuk_carpim_3([-10, -9, 1, 2, 3]) == 270
assert en_buyuk_carpim_3([1, 2, 3, 4, 5])    == 60
print("✅ Tüm testler geçti.")


---

### 📝 Quiz Cevap Anahtarı

| Soru | Cevap | Açıklama |
|---|---|---|
| Q1 | **C** | Dijkstra her adımda "en yakın ziyaret edilmemiş düğümü" seçer — greedy. |
| Q2 | **Yanlış (Y)** | Sadece bazı problemlerde optimum; 0/1 sırt çantası, TSP gibi problemlerde değil. |
| Q3 | "**En erken biten** dersi seç, çakışanları at, tekrar et." | Provably-optimum greedy strateji. |
| Q4 | **B** | 0/1 sırt çantası NP-tam; doğru çözüm DP. |
| Q5 | **Doğru (D)** | BFS seviye seviye genişlediği için ağırlıksız grafta en kısa yolu garanti eder. |

---

### 🎓 Hafta 8 Tamam!
- BFS/DFS'i tazeledik.
- Greedy mantığını dört temel örnekle (dizi max, çizelgeleme, kesirli sırt çantası, set covering) gördük.
- "Greedy ne zaman çalışır, ne zaman çalışmaz" sezgisini kazandık.
- **Sonraki hafta:** Dinamik programlama (dynamic programming) — greedy'nin başarısız olduğu yerlerde devreye giren güçlü teknik.
